# 1 — Where the data comes from

**Question this project answers:** which UK companies should a land-sourcing and planning-intelligence
product approach first, and what makes them a better prospect than the ones ranked below?

The worked example is a product sold to **UK property developers** — the segment LandTech, Orbital and
Searchland compete in. Naming the vendor matters. A lead score is only defensible relative to a
product: "good prospect" means nothing on its own, and every weighting decision in notebook 2 is an
argument about *this* buyer.

---

## Why the bulk download and not the API

Companies House publishes the same register two ways.

| | Free Company Data Product | REST API |
|---|---|---|
| Access | Public download, no key | Free key, rate limited |
| Shape | Monthly CSV snapshot, ~5.7m rows | Per-company lookups and search |
| Reproducible by a reader | Yes, immediately | Only with their own key |

The API is the better tool for enriching a handful of companies you have already chosen. It is the
wrong tool for *defining a segment*, because the segment is the thing being discovered — the question
is which companies exist, not what is true about a known list. And a pipeline a reader cannot re-run
is a claim rather than evidence.

**Snapshot used: `BasicCompanyDataAsOneFile-2026-09-01`** — 493 MB zipped, roughly 2 GB as CSV,
5,689,367 companies.

## The three gates

`get_data.py` streams the archive in 250,000-row chunks and keeps a company only if it passes all
three:

1. **Status is Active.** Dissolved and liquidating companies are not prospects.
2. **Registered in a London postcode area** — E, EC, N, NW, SE, SW, W, WC.
3. **Declares at least one relevant SIC code** — the graded list in `config.SIC_RELEVANCE`.

The gates decide *membership*; the score in notebook 2 decides *order*. Keeping them apart is why
changing a weight is a five-second re-run while changing the segment is a two-gigabyte one.

Only 18 of the file's 55 columns are read. Naming them explicitly holds peak memory in the low
hundreds of MB and documents exactly what the score is allowed to see.

In [1]:
import pandas as pd

import config
import get_data

print("Vendor:  ", config.VENDOR_PROFILE)
print("Snapshot:", config.SNAPSHOT_DATE)
print("Source:  ", config.BULK_URL)
print()
print("SIC codes in scope, by relevance:")
for sic_code, relevance in sorted(config.SIC_RELEVANCE.items(), key=lambda kv: -kv[1]):
    print(f"  {sic_code}   {relevance:.2f}")

Vendor:   Land sourcing and planning intelligence, sold to UK property developers
Snapshot: 2026-09-01
Source:   https://download.companieshouse.gov.uk/BasicCompanyDataAsOneFile-2026-09-01.zip

SIC codes in scope, by relevance:
  41100   1.00
  68100   0.70
  41201   0.55
  41202   0.55
  68209   0.35
  42990   0.25


### Postcode handling

Registered address is the only location signal the free dataset carries. `postcode_area` takes the
alphabetic head of the outward code, which is what identifies the London postal district group.

In [2]:
for example in ["SW6 5BP", "EC2R 5AA", "W1J 8DX", "M1 4BT", "", None]:
    area = get_data.postcode_area(example)
    verdict = "in scope" if area in config.LONDON_POSTCODE_AREAS else "excluded"
    print(f"{str(example):10} -> area {area or '(none)':6} {verdict}")

SW6 5BP    -> area SW     in scope
EC2R 5AA   -> area EC     in scope
W1J 8DX    -> area W      in scope
M1 4BT     -> area M      excluded
           -> area (none) excluded
None       -> area (none) excluded


## What came back

Run `python get_data.py` once before this notebook. It writes `data/segment.csv`, which is small
enough to iterate on in a second.

In [3]:
segment = pd.read_csv(config.SEGMENT_CSV, dtype=str)

print(f"{len(segment):,} companies in segment, from 5,689,367 scanned "
      f"({len(segment) / 5_689_367:.2%} of the register)")
print()
print("Most common declared primary activity:")
print(segment["SICCode.SicText_1"].value_counts().head(6).to_string())

156,107 companies in segment, from 5,689,367 scanned (2.74% of the register)

Most common declared primary activity:
SICCode.SicText_1
68100 - Buying and selling of own real estate                       56487
68209 - Other letting and operating of own or leased real estate    56237
41100 - Development of building projects                            20714
41202 - Construction of domestic buildings                           8225
41201 - Construction of commercial buildings                         6070
42990 - Construction of other civil engineering projects n.e.c.      1174


### A limitation worth stating before someone else finds it

**Registered address is not trading address.** A developer building in Corby can be registered at its
accountant's office in Mayfair, and thousands of companies use formation agents as their registered
office. This is therefore a list of *London-registered* developers, not *London-building* ones.

For this vendor the distinction is mostly acceptable — the software is bought by a head-office team,
and head office is where the post goes. It would not be acceptable for a product sold to a site
manager. The specific failure mode it creates is dealt with directly in notebook 3.